In [16]:
import pandas as pd
import panel as pn
import numpy as np
from sqlalchemy import create_engine
from datetime import date, timedelta
import calendar
from pandas.tseries.offsets import BDay
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set

pn.extension('tabulator')
pd.options.display.max_rows = 11

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()
engine = create_engine("sqlite:///c:\\ruby\\portmy\\db\\development.sqlite3")
conmy = engine.connect()

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
osd_path = "\\Users\\User\\OneDrive\\Documents\\obsidian-git-sync\\Data\\"

today = date.today()
today

datetime.date(2023, 2, 5)

### Set today = last closed business day

In [2]:
num_business_days = BDay(1)
yesterday = today - num_business_days
yesterday = yesterday.date()
today, yesterday

(datetime.date(2023, 2, 5), datetime.date(2023, 2, 3))

### Restart and Run All Cells

In [3]:
sqlUpd = """
UPDATE buy B
SET dividend =
(SELECT DIVIDEND FROM dividend D
WHERE B.name = D.name)"""
rp = const.execute(sqlUpd)
rp.rowcount

27

In [4]:
cols = 'name period shares mkt_price cost_amt base_amt pct'.split()
colt = 'name shares unit_cost cost_amt price market_amt mkt_pct div_rnk amt_rnk div_amt'.split()
colu = 'name shares unit_cost cost_amt dividend div_amt cost_pct period'.split()
colv = 'name shares unit_cost cost_amt mkt_price market_amt dividend div_amt cost_pct mkt_pct profit pft_pct'.split()
colw = 'name cost_amt market_amt div_amt cost_pct mkt_pct pft_pct'.split()
colx = 'name shares unit_cost cost_amt mkt_price market_amt dividend div_amt cost_pct mkt_pct profit pft_pct period'.split()

In [5]:
format_dict = {
    'shares':'{:,}',  
    'mkt_price':'฿{:.2f}','unit_cost':'฿{:.2f}',
    'cost_amt':'฿{:,.2f}','div_amt':'฿{:,.2f}','market_amt':'฿{:,.2f}','profit':'฿{:,.2f}','base_amt':'฿{:,.2f}',
    'dividend':'฿{:.4f}',    
    'pct':'{:,.2f}%','cost_pct':'{:,.2f}%','mkt_pct':'{:,.2f}%','pft_pct':'{:,.2f}%',
}

### Process portfolio (table buy in mysql stock)

In [6]:
sql = """
SELECT name, volbuy, price AS unit_cost, dividend, period
FROM buy
WHERE active = 1"""
print(sql)


SELECT name, volbuy, price AS unit_cost, dividend, period
FROM buy
WHERE active = 1


In [7]:
buy = pd.read_sql(sql, const)
buy.rename(columns={'volbuy':'shares'},inplace=True)
buy['shares'] = buy['shares'].astype('int64')
buy['cost_amt'] = round(buy['shares'] * buy['unit_cost'], 2)
buy['cost_pct'] = round(buy['dividend'] / buy['unit_cost'] * 100, 2)
buy['div_amt'] = round(buy['shares'] * buy['dividend'], 2)
buy[colu].sort_values('cost_pct',ascending=False).style.format(format_dict)

,name,shares,unit_cost,cost_amt,dividend,div_amt,cost_pct,period
24,RCL,"27,000",฿38.75,"฿1,046,250.00",฿7.0000,"฿189,000.00",18.06%,2
11,JASIF,"130,000",฿10.00,"฿1,300,000.00",฿0.9400,"฿122,200.00",9.40%,2
18,GVREIT,"40,000",฿8.90,"฿356,000.00",฿0.7786,"฿31,144.00",8.75%,2
5,TMT,"36,000",฿10.20,"฿367,200.00",฿0.8500,"฿30,600.00",8.33%,1
15,WHAIR,"50,000",฿8.70,"฿435,000.00",฿0.6744,"฿33,720.00",7.75%,2
7,WHART,"30,000",฿11.70,"฿351,000.00",฿0.8941,"฿26,823.00",7.64%,2
9,SENA,"105,000",฿4.48,"฿470,400.00",฿0.3384,"฿35,532.00",7.55%,2
12,ASP,"30,000",฿3.80,"฿114,000.00",฿0.2700,"฿8,100.00",7.11%,1
4,DIF,"40,000",฿14.70,"฿588,000.00",฿1.0335,"฿41,340.00",7.03%,2
8,BCH,"15,000",฿21.46,"฿321,900.00",฿1.4000,"฿21,000.00",6.52%,2


In [8]:
file_name = 'hi-dividend.csv'
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name

buy[colu].sort_values(['cost_pct'],ascending=[False]).to_csv(output_file, index=False)
buy[colu].sort_values(['cost_pct'],ascending=[False]).to_csv(data_file, index=False)
buy[colu].sort_values(['cost_pct'],ascending=[False]).to_csv(box_file, index=False)
buy[colu].sort_values(['cost_pct'],ascending=[False]).to_csv(one_file, index=False)

### Start of Period Calculation

In [11]:
sql = '''
SELECT B.name, volbuy, B.price AS unit_cost, 
dividend, P.price AS mkt_price, period
FROM buy B JOIN price P
ON B.name = P.name
WHERE P.date = "%s"
AND active = 1'''
sql = sql % yesterday 
print(sql)


SELECT B.name, volbuy, B.price AS unit_cost, 
dividend, P.price AS mkt_price, period
FROM buy B JOIN price P
ON B.name = P.name
WHERE P.date = "2023-02-03"
AND active = 1


In [12]:
df = pd.read_sql(sql, const)
df.rename(columns={'volbuy':'shares'},inplace=True)
df['shares'] = df.shares.astype(int)
df['cost_amt'] = round(df['shares'] * df['unit_cost'],2)
df['market_amt'] = round(df['shares'] * df['mkt_price'],2)
df['div_amt'] = round(df['shares'] * df['dividend'],2)
df['cost_pct'] = round(df['dividend'] / df['unit_cost'] * 100,2)
df['mkt_pct'] = round(df['dividend'] / df['mkt_price'] * 100,2)
df['profit'] = round((df['mkt_price'] - df['unit_cost']) * df['shares'],2)
df['pft_pct'] = round(df['profit'] / df['cost_amt'] * 100,2)
df[colv].sort_values(['pft_pct'],ascending=[False]).style.format(format_dict)

,name,shares,unit_cost,cost_amt,mkt_price,market_amt,dividend,div_amt,cost_pct,mkt_pct,profit,pft_pct
25,TFFIF,"10,000",฿7.50,"฿75,000.00",฿8.25,"฿82,500.00",฿0.3721,"฿3,721.00",4.96%,4.51%,"฿7,500.00",10.00%
18,GVREIT,"40,000",฿8.90,"฿356,000.00",฿9.75,"฿390,000.00",฿0.7786,"฿31,144.00",8.75%,7.99%,"฿34,000.00",9.55%
16,ORI,"45,000",฿11.00,"฿495,000.00",฿12.00,"฿540,000.00",฿0.5700,"฿25,650.00",5.18%,4.75%,"฿45,000.00",9.09%
1,SINGER,"3,600",฿28.00,"฿100,800.00",฿28.25,"฿101,700.00",฿0.8500,"฿3,060.00",3.04%,3.01%,฿900.00,0.89%
7,WHART,"30,000",฿11.70,"฿351,000.00",฿11.80,"฿354,000.00",฿0.8941,"฿26,823.00",7.64%,7.58%,"฿3,000.00",0.85%
8,BCH,"15,000",฿21.46,"฿321,900.00",฿21.50,"฿322,500.00",฿1.4000,"฿21,000.00",6.52%,6.51%,฿600.00,0.19%
26,CPNREIT,"30,000",฿19.10,"฿573,000.00",฿19.10,"฿573,000.00",฿0.9059,"฿27,177.00",4.74%,4.74%,฿0.00,0.00%
21,DCC,"60,000",฿2.96,"฿177,600.00",฿2.86,"฿171,600.00",฿0.1600,"฿9,600.00",5.41%,5.59%,"฿-6,000.00",-3.38%
13,IVL,"4,800",฿42.00,"฿201,600.00",฿40.50,"฿194,400.00",฿1.4500,"฿6,960.00",3.45%,3.58%,"฿-7,200.00",-3.57%
6,JMT,"3,300",฿57.00,"฿188,100.00",฿54.75,"฿180,675.00",฿0.8200,"฿2,706.00",1.44%,1.50%,"฿-7,425.00",-3.95%


In [13]:
df.nlargest(5, 'mkt_pct')[colw].style.format(format_dict)

,name,cost_amt,market_amt,div_amt,cost_pct,mkt_pct,pft_pct
24,RCL,"฿1,046,250.00","฿830,250.00","฿189,000.00",18.06%,22.76%,-20.65%
11,JASIF,"฿1,300,000.00","฿1,072,500.00","฿122,200.00",9.40%,11.39%,-17.50%
5,TMT,"฿367,200.00","฿295,200.00","฿30,600.00",8.33%,10.37%,-19.61%
12,ASP,"฿114,000.00","฿93,000.00","฿8,100.00",7.11%,8.71%,-18.42%
9,SENA,"฿470,400.00","฿424,200.00","฿35,532.00",7.55%,8.38%,-9.82%


In [14]:
df.nsmallest(5, 'mkt_pct')[colw].style.format(format_dict)

,name,cost_amt,market_amt,div_amt,cost_pct,mkt_pct,pft_pct
6,JMT,"฿188,100.00","฿180,675.00","฿2,706.00",1.44%,1.50%,-3.95%
23,SCC,"฿243,000.00","฿202,800.00","฿4,800.00",1.98%,2.37%,-16.54%
1,SINGER,"฿100,800.00","฿101,700.00","฿3,060.00",3.04%,3.01%,0.89%
2,KCE,"฿894,000.00","฿696,000.00","฿24,000.00",2.68%,3.45%,-22.15%
13,IVL,"฿201,600.00","฿194,400.00","฿6,960.00",3.45%,3.58%,-3.57%


In [17]:
file_name = 'buy-div-price.csv'
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
osd_file = osd_path + file_name

df[colx].sort_values(['period','name'],ascending=[True,True]).to_csv(output_file)
df[colx].sort_values(['period','name'],ascending=[True,True]).to_csv(data_file)
df[colx].sort_values(['period','name'],ascending=[True,True]).to_csv(box_file)
df[colx].sort_values(['period','name'],ascending=[True,True]).to_csv(osd_file)

### Short term stocks

In [18]:
short_term = df.period == '4'
df.loc[short_term][colv].sort_values(['pft_pct'],ascending=[False]).style.format(format_dict)

,name,shares,unit_cost,cost_amt,mkt_price,market_amt,dividend,div_amt,cost_pct,mkt_pct,profit,pft_pct
1,SINGER,"3,600",฿28.00,"฿100,800.00",฿28.25,"฿101,700.00",฿0.8500,"฿3,060.00",3.04%,3.01%,฿900.00,0.89%
6,JMT,"3,300",฿57.00,"฿188,100.00",฿54.75,"฿180,675.00",฿0.8200,"฿2,706.00",1.44%,1.50%,"฿-7,425.00",-3.95%
10,BANPU,"18,000",฿12.00,"฿216,000.00",฿11.50,"฿207,000.00",฿0.7000,"฿12,600.00",5.83%,6.09%,"฿-9,000.00",-4.17%
20,JMART,"3,600",฿39.50,"฿142,200.00",฿37.25,"฿134,100.00",฿1.4600,"฿5,256.00",3.70%,3.92%,"฿-8,100.00",-5.70%
23,SCC,600,฿405.00,"฿243,000.00",฿338.00,"฿202,800.00",฿8.0000,"฿4,800.00",1.98%,2.37%,"฿-40,200.00",-16.54%


In [19]:
p4cost = df.loc[short_term].cost_amt.sum()
p4profit = df.loc[short_term].profit.sum()
p4pct = round(p4profit/p4cost*100, 2)
p4cost, p4profit, p4pct, df.loc[short_term].shape[0]

(890100.0, -63825.0, -7.17, 5)

In [20]:
array = pd.Series([p4cost, p4profit, p4pct])
array = array.map('฿{:,.2f}'.format)
for value in array:
    print(f"The value is: {value}")

The value is: ฿890,100.00
The value is: ฿-63,825.00
The value is: ฿-7.17


### Long term stocks

In [21]:
long_term = df.period == '3'
df[long_term].sort_values(['pft_pct'],ascending=[False]).shape

(5, 13)

In [22]:
df[long_term].nlargest(5, 'pft_pct')[['name','cost_amt','market_amt','profit','pft_pct']].style.format(format_dict)

,name,cost_amt,market_amt,profit,pft_pct
16,ORI,"฿495,000.00","฿540,000.00","฿45,000.00",9.09%
13,IVL,"฿201,600.00","฿194,400.00","฿-7,200.00",-3.57%
19,NER,"฿201,150.00","฿172,800.00","฿-28,350.00",-14.09%
2,KCE,"฿894,000.00","฿696,000.00","฿-198,000.00",-22.15%
22,SYNEX,"฿430,500.00","฿249,000.00","฿-181,500.00",-42.16%


In [23]:
p3cost = df.loc[long_term].cost_amt.sum()
p3profit = df.loc[long_term].profit.sum()
p3pct = round(p3profit/p3cost*100, 2)
p3cost, p3profit, p3pct,df.loc[long_term].shape[0]

(2222250.0, -370050.0, -16.65, 5)

In [24]:
array = pd.Series([p3cost, p3profit, p3pct])
array = array.map('฿{:,.2f}'.format)
for value in array:
    print(f"The value is: {value}")

The value is: ฿2,222,250.00
The value is: ฿-370,050.00
The value is: ฿-16.65


### High dividend stocks

In [25]:
hi_dividend = df.period == '2'
df.loc[hi_dividend][colv].sort_values(['mkt_pct'],ascending=[False]).style.format(format_dict)

,name,shares,unit_cost,cost_amt,mkt_price,market_amt,dividend,div_amt,cost_pct,mkt_pct,profit,pft_pct
24,RCL,"27,000",฿38.75,"฿1,046,250.00",฿30.75,"฿830,250.00",฿7.0000,"฿189,000.00",18.06%,22.76%,"฿-216,000.00",-20.65%
11,JASIF,"130,000",฿10.00,"฿1,300,000.00",฿8.25,"฿1,072,500.00",฿0.9400,"฿122,200.00",9.40%,11.39%,"฿-227,500.00",-17.50%
9,SENA,"105,000",฿4.48,"฿470,400.00",฿4.04,"฿424,200.00",฿0.3384,"฿35,532.00",7.55%,8.38%,"฿-46,200.00",-9.82%
15,WHAIR,"50,000",฿8.70,"฿435,000.00",฿8.05,"฿402,500.00",฿0.6744,"฿33,720.00",7.75%,8.38%,"฿-32,500.00",-7.47%
18,GVREIT,"40,000",฿8.90,"฿356,000.00",฿9.75,"฿390,000.00",฿0.7786,"฿31,144.00",8.75%,7.99%,"฿34,000.00",9.55%
4,DIF,"40,000",฿14.70,"฿588,000.00",฿13.60,"฿544,000.00",฿1.0335,"฿41,340.00",7.03%,7.60%,"฿-44,000.00",-7.48%
7,WHART,"30,000",฿11.70,"฿351,000.00",฿11.80,"฿354,000.00",฿0.8941,"฿26,823.00",7.64%,7.58%,"฿3,000.00",0.85%
8,BCH,"15,000",฿21.46,"฿321,900.00",฿21.50,"฿322,500.00",฿1.4000,"฿21,000.00",6.52%,6.51%,฿600.00,0.19%
17,SCCC,"1,200",฿171.00,"฿205,200.00",฿158.50,"฿190,200.00",฿9.0000,"฿10,800.00",5.26%,5.68%,"฿-15,000.00",-7.31%
21,DCC,"60,000",฿2.96,"฿177,600.00",฿2.86,"฿171,600.00",฿0.1600,"฿9,600.00",5.41%,5.59%,"฿-6,000.00",-3.38%


In [26]:
df[hi_dividend].nlargest(5, 'mkt_pct')[['name','cost_amt','market_amt','div_amt','cost_pct','mkt_pct']]\
.style.format(format_dict)

,name,cost_amt,market_amt,div_amt,cost_pct,mkt_pct
24,RCL,"฿1,046,250.00","฿830,250.00","฿189,000.00",18.06%,22.76%
11,JASIF,"฿1,300,000.00","฿1,072,500.00","฿122,200.00",9.40%,11.39%
9,SENA,"฿470,400.00","฿424,200.00","฿35,532.00",7.55%,8.38%
15,WHAIR,"฿435,000.00","฿402,500.00","฿33,720.00",7.75%,8.38%
18,GVREIT,"฿356,000.00","฿390,000.00","฿31,144.00",8.75%,7.99%


In [27]:
p2profit = df.loc[hi_dividend].profit.sum()
p2cost = df.loc[hi_dividend].cost_amt.sum()
p2dividend = df.loc[hi_dividend].div_amt.sum()
p2yield = round(p2profit/p2cost*100,2)
p2cost, p2profit, p2yield, p2dividend, df.loc[hi_dividend].shape[0]

(5899350.0, -542100.0, -9.19, 552057.0, 12)

In [28]:
array = pd.Series([p2cost, p2profit, p2yield, p2dividend])
array = array.map('฿{:,.2f}'.format)
for value in array:
    print(f"The value is: {value}")

The value is: ฿5,899,350.00
The value is: ฿-542,100.00
The value is: ฿-9.19
The value is: ฿552,057.00


### Disposal stocks

In [29]:
disposal = df.period == '1'
df.loc[disposal][colv].sort_values(['pft_pct'],ascending=[True]).style.format(format_dict)

,name,shares,unit_cost,cost_amt,mkt_price,market_amt,dividend,div_amt,cost_pct,mkt_pct,profit,pft_pct
0,STA,"7,500",฿39.25,"฿294,375.00",฿22.30,"฿167,250.00",฿1.5500,"฿11,625.00",3.95%,6.95%,"฿-127,125.00",-43.18%
3,MCS,"75,000",฿15.40,"฿1,155,000.00",฿9.95,"฿746,250.00",฿0.5000,"฿37,500.00",3.25%,5.03%,"฿-408,750.00",-35.39%
14,PTTGC,"6,000",฿64.75,"฿388,500.00",฿49.50,"฿297,000.00",฿2.5000,"฿15,000.00",3.86%,5.05%,"฿-91,500.00",-23.55%
5,TMT,"36,000",฿10.20,"฿367,200.00",฿8.20,"฿295,200.00",฿0.8500,"฿30,600.00",8.33%,10.37%,"฿-72,000.00",-19.61%
12,ASP,"30,000",฿3.80,"฿114,000.00",฿3.10,"฿93,000.00",฿0.2700,"฿8,100.00",7.11%,8.71%,"฿-21,000.00",-18.42%


In [30]:
p1cost = df.loc[disposal].cost_amt.sum()
p1profit = df.loc[disposal].profit.sum()
p1pct = round(p1profit/p1cost*100,2)
p1cost, p1profit, p1pct,df.loc[disposal].shape[0]

(2319075.0, -720375.0, -31.06, 5)

In [31]:
array = pd.Series([p1cost, p1profit, p1pct])
array = array.map('฿{:,.2f}'.format)
for value in array:
    print(f"The value is: {value}")

The value is: ฿2,319,075.00
The value is: ฿-720,375.00
The value is: ฿-31.06
